# Claude Haiku 5.5 as the claim extractor (EXP-24)

DEC-24 chose four reasoner models. Verifying their outputs with today's claim extractor (Claude Sonnet 5.5) would cost about $150 against about $96 left, so this checks whether **Claude Haiku 5.5** extracts claims well enough to replace it:
- **EXP-11**, the seeded-error set (5 gold outputs, 70 planted errors): the bar is recall of at least 0.80 for every sub-type and false flags of at most 0.10 (`docs/HALLUCINATION_METRIC.md` §8).
- **EXP-12**, the five fixtures: compared with Sonnet's extraction under the same judge (`judge_v3`, EXP-23: 19 of 350 claims flagged).

Each runs in its own folder under `runs/EXP-24_haiku_extractor/`, seeded with copies of the original judge and retrieval caches; the EXP-11 and EXP-12 folders are not changed. The logic is in `src/strokecopilot/nlp/verifier/recheck.py` (`extractor_check`). Estimated cost about $0.50-1.00 (not measured): 80 Haiku extractions and the judge calls for new claim texts. No reasoner calls. Everything is synthetic; nothing here touches patient data.

**Before you start**
- Use a CPU runtime: Runtime → Change runtime type → None.
- In Secrets (the key icon), `ANTHROPIC_API_KEY` must have notebook access on.

**Run it:** Runtime → Run all, and allow Drive access when asked. Allow about 30 minutes. If Colab disconnects, Run all again: the caches make it resume. Send back the line that starts with `OK:` and the two lines above it.

## 1. Settings

In [ ]:
EXPERIMENT, NAME = 'EXP-24', 'EXP-24_haiku_extractor'
ROOT = '/content/drive/MyDrive/StrokeCopilot'
REPO_REF = 'main'
RUN_DIR = f'{ROOT}/runs/{NAME}'
EXP11_DIR, EXP12_DIR = f'{ROOT}/runs/EXP-11_judge_bakeoff', f'{ROOT}/runs/EXP-12_end_to_end_fixtures'
EXTRACTOR = ('claude-haiku-5-5', 'medium')
print(f'{EXPERIMENT}: claim extractor {EXTRACTOR[0]} (effort {EXTRACTOR[1]}) on EXP-11 and EXP-12.')

## 2. Install the package and mount Drive
The package is pinned to one commit. Neither key is printed or kept in a variable.

In [ ]:
import json, os, subprocess, sys
from google.colab import drive, userdata

if any(m == 'strokecopilot' or m.startswith('strokecopilot.') for m in sys.modules):
    raise RuntimeError('This session already loaded an older copy of the package. Use Runtime → Restart session, then Run all.')

repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'strokecopilot[llm,retrieval] @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
# pip treats the same version as already installed, so the pinned commit is forced in
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'strokecopilot @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
print(f'Package installed at commit {COMMIT[:7]}.')

try:
    has_key = bool(userdata.get('ANTHROPIC_API_KEY'))
except Exception:
    has_key = False
if not has_key:
    raise RuntimeError('ANTHROPIC_API_KEY is not in Secrets, or notebook access is off.')
del has_key

drive.mount('/content/drive')
if os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise RuntimeError(f'{RUN_DIR} already holds a finished run. Rename it on Drive before running again.')
os.makedirs(RUN_DIR, exist_ok=True)


## 3. Verify again with the Haiku extractor
`config.json` is written first.

In [ ]:
import json
from strokecopilot.nlp.corpus import index, ingest
from strokecopilot.nlp.reasoner import providers
from strokecopilot.nlp.verifier import recheck as rc, trace as tr

config = {'experiment': EXPERIMENT, 'commit': COMMIT, 'extractor': {'model': EXTRACTOR[0], 'effort': EXTRACTOR[1], 'max_tokens': 16000},
          'judge': {'model': tr.JUDGE_MODEL, 'effort': tr.JUDGE_EFFORT, 'prompt': tr.JUDGE_PROMPT}, 'tau': tr.TAU, 'top_k': tr.TOP_K,
          'corpus_sha256': ingest.version(), 'sources': ['EXP-11_judge_bakeoff', 'EXP-12_end_to_end_fixtures'],
          'data_version': 'synthetic fixtures and seeded errors'}
with open(f'{RUN_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=1)
retriever = index.Index.from_corpus()
extractor = lambda: providers.ClaudeJSON(EXTRACTOR[0], effort=EXTRACTOR[1], max_tokens=16000)
judge = lambda: providers.ClaudeJSON(tr.JUDGE_MODEL, effort=tr.JUDGE_EFFORT, max_tokens=8000)
result = rc.extractor_check(EXP11_DIR, EXP12_DIR, RUN_DIR, extractor, judge, retriever, extra={'experiment': EXPERIMENT, 'commit': COMMIT})

## 4. Summary

In [ ]:
b, f = result['exp11'], result['exp12']
cost = b['extractor_cost_usd_estimate'] + f['extractor_cost_usd_estimate'] + b['cost_usd_estimate'] + f['cost_usd_estimate']
print(f"EXP-11 recall {b['recall']}; missed {b['missed']}")
print(f"EXP-12 sub-types {f['subtypes']}")
print(f"OK: {EXPERIMENT} commit {COMMIT[:7]}: {result['extractor']} extractor; EXP-11 bar {'met' if b['meets_bar'] else 'NOT met'} "
      f"(min recall {b['min_recall']:.2f}, false flags {b['false_flag_rate']:.3f}); EXP-12 rate {f['hallucination_rate']['value']:.3f} "
      f"({f['hallucination_rate']['numerator']}/{f['claims']}), critical {f['critical_hallucination_rate']['value']:.3f}; "
      f"extractor ${b['extractor_cost_usd_estimate'] + f['extractor_cost_usd_estimate']:.3f}; about ${cost:.2f}. Saved to runs/{NAME}/.")